In [1]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import re
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_recall_curve,
    classification_report
)

# 可选：LightGBM 更强
import lightgbm as lgb

RANDOM_STATE = 42

In [2]:

# =========================
# 1) 读取主表与距离映射
# =========================
df = pd.read_csv("panel_data_1111/after_merge_1111.csv")
dist_bj_sh = pd.read_csv("dist_origin_to_bj_sh.csv")  # 只含 hs_residence, dist_to_bj, dist_to_sh

# 仅保留跨省流动样本
df = df[df["Migrate"] == 1].copy()
df = df[df["hs_residence"] != df["pro_code"]].copy()
if "Migrate_1" in df.columns:
    df = df[df["Migrate_1"] == 1].copy()

# 标签：是否去北京/上海
df["y_bjsh"] = df["pro_code"].isin([11, 31]).astype(int)

In [3]:

# =========================
# 2) 去除泄漏变量
# =========================
drop_cols_leak = [
    "pro_code","pro_name","city_clean","is_beijing","is_shanghai",
    "pro_name_true","English_name","gdp_after_move","migration_distance_km"
]
drop_cols_leak += [c for c in df.columns if c.endswith("_a")]
df = df.drop(columns=[c for c in drop_cols_leak if c in df.columns])

In [4]:

# =========================
# 3) 合并“原籍→北京/上海”距离（常量，不泄漏）
# =========================
df["hs_residence"] = pd.to_numeric(df["hs_residence"], errors="coerce").astype("Int64")
dist_bj_sh["hs_residence"] = pd.to_numeric(dist_bj_sh["hs_residence"], errors="coerce").astype("Int64")
df = df.merge(dist_bj_sh, on="hs_residence", how="left")

# 衍生距离特征（不泄漏）
df["dist_bj_minus_sh"] = df["dist_to_bj"] - df["dist_to_sh"]
df["dist_min_bj_sh"]   = df[["dist_to_bj","dist_to_sh"]].min(axis=1)
df["dist_ratio_bj_over_sh"] = (df["dist_to_bj"] + 1) / (df["dist_to_sh"] + 1)

In [5]:

# =========================
# 4) 静态宏观差值：北京/上海静态值 - 原籍_b
# =========================
# external_data2.xlsx：你的静态宏观来源（和之前 on="hs_residence" 的那张）
# 若文件不存在或列名不一致，可跳过本段（保持鲁棒）
def safe_name(name: str) -> str:
    return re.sub(r"[^0-9a-zA-Z_]+", "_", str(name)).strip("_").lower()

macro_b_cols = [c for c in df.columns if c.endswith("_b")]
try:
    ext2 = pd.read_excel("external_data2.xlsx")
    # 兼容 ext2 使用 hs_residence 或 pro_code 作为省份列
    key_col = "hs_residence" if "hs_residence" in ext2.columns else ("pro_code" if "pro_code" in ext2.columns else None)
    if key_col is not None:
        row_bj = ext2.loc[ext2[key_col] == 11]
        row_sh = ext2.loc[ext2[key_col] == 31]
        if not row_bj.empty and not row_sh.empty:
            row_bj = row_bj.iloc[0]
            row_sh = row_sh.iloc[0]
            for col in macro_b_cols:
                base = col[:-2]  # 去掉 _b
                if base in ext2.columns:
                    v_bj = row_bj[base]
                    v_sh = row_sh[base]
                    df[f"delta_{safe_name(base)}_bj"] = v_bj - df[col]
                    df[f"delta_{safe_name(base)}_sh"] = v_sh - df[col]
        else:
            print("[warn] external_data2.xlsx 中未找到北京/上海两行，跳过静态差值构造。")
    else:
        print("[warn] external_data2.xlsx 中没有 hs_residence/pro_code 键，跳过静态差值构造。")
except FileNotFoundError:
    print("[warn] 未找到 external_data2.xlsx，跳过静态差值构造。")

In [6]:

# =========================
# 5) 年度 GDP 差值（可选，若有年度面板；无则自动跳过）
# =========================
# 用年度面板获得北京/上海当年 GDP，再与 gdp_before_move 做 log 差（非必须）
try:
    gdp_panel = pd.read_excel("china_correct_panel.xlsx")  # 需要包含列：pro_code, year, real_GDP
    gdp_panel = gdp_panel.rename(columns={"year":"migration_year"})
    bj = gdp_panel[gdp_panel["pro_code"] == 11][["migration_year","real_GDP"]].rename(columns={"real_GDP":"gdp_bj"})
    sh = gdp_panel[gdp_panel["pro_code"] == 31][["migration_year","real_GDP"]].rename(columns={"real_GDP":"gdp_sh"})
    df = df.merge(bj, on="migration_year", how="left")
    df = df.merge(sh, on="migration_year", how="left")

    if "gdp_before_move" in df.columns:
        df["log_gdp_before"] = np.log(df["gdp_before_move"].replace({0: np.nan}))
        df["log_gdp_bj"] = np.log(df["gdp_bj"].replace({0: np.nan}))
        df["log_gdp_sh"] = np.log(df["gdp_sh"].replace({0: np.nan}))
        df["delta_log_gdp_bj_vs_origin"] = df["log_gdp_bj"] - df["log_gdp_before"]
        df["delta_log_gdp_sh_vs_origin"] = df["log_gdp_sh"] - df["log_gdp_before"]
    else:
        print("[info] 未找到 gdp_before_move，跳过年度 GDP 差值。")
except FileNotFoundError:
    print("[info] 未找到 china_correct_panel.xlsx，跳过年度 GDP 差值。")


In [7]:

# =========================
# 6) 个人特征与金额变换
# =========================
# 金额类 log1p
for c in ["income_total_m_win","exp_total_m_win","food_exp_m_win","income_to_home_win","rent_m_win"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
        df[f"log1p_{c}"] = np.log1p(df[c])

# 构建特征清单
label = "y_bjsh"
cat_cols = []
if "employment_group" in df.columns:
    cat_cols.append("employment_group")

# 不将 hs_residence 用作特征（仅用于合并映射）
exclude_cols = {label, "hs_residence", "gdp_bj", "gdp_sh", "log_gdp_before", "log_gdp_bj", "log_gdp_sh"}
feature_cols = [c for c in df.columns if c not in exclude_cols]

# 可选：去掉高缺失（>80%）的列，减少噪声
na_rate = df[feature_cols].isna().mean()
drop_high_na = na_rate[na_rate > 0.8].index.tolist()
if drop_high_na:
    print(f"[info] 丢弃高缺失特征（>{0.8:.0%}）：{len(drop_high_na)} 列")
    feature_cols = [c for c in feature_cols if c not in drop_high_na]

In [8]:

# =========================
# 7) 随机划分（打乱 + Stratified）
# =========================
X = df[feature_cols].copy()
y = df[label].astype(int).values

# 70% 训练，15% 验证，15% 测试
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_va, X_te, y_va, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_tmp
)

In [9]:

# =========================
# 8) 预处理与基线模型：L1 Logistic
# =========================
from pandas.api.types import is_numeric_dtype

# 先得到 feature_cols（保持你原有逻辑）
# feature_cols = [...]

# 1) 可选：自动把“看起来像数值”的 object 列转为数值
obj_cols = df[feature_cols].select_dtypes(include=["object"]).columns.tolist()
def coerce_numeric_like(df, cols, min_numeric_rate=0.9):
    for c in cols:
        s = df[c].astype(str).str.replace(",", "").str.strip()
        s_num = pd.to_numeric(s, errors="coerce")
        # 若 >=90% 的值能成功转成数值，则视为数值列
        if s_num.notna().mean() >= min_numeric_rate:
            df[c] = s_num
coerce_numeric_like(df, obj_cols)

# 2) 依据 dtype 动态划分 数值/类别 列（避免把字符串放进“数值通道”）
num_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]

print(f"[info] 数值特征数: {len(num_cols)}, 类别特征数: {len(cat_cols)}")
# 若某些你明确知道是类别列（如 employment_group）不在 cat_cols，可以手工补充：
for must_cat in ["employment_group"]:
    if must_cat in feature_cols and must_cat not in cat_cols:
        if must_cat in num_cols:
            num_cols.remove(must_cat)
        cat_cols.append(must_cat)

# 3) 构建 ColumnTransformer（如果某一类为空也能正常工作）
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

transformers = []
if len(num_cols) > 0:
    transformers.append(("num", SimpleImputer(strategy="median"), num_cols))
if len(cat_cols) > 0:
    transformers.append((
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]),
        cat_cols
    ))

preprocess = ColumnTransformer(transformers=transformers, remainder="drop")

[info] 数值特征数: 92, 类别特征数: 2


In [10]:
X_te.columns

Index(['male', 'birth_year', 'age', 'is_han', 'high_school', 'junior_college',
       'bachelor', 'graduate', 'rural', 'Migrate', 'Migrate_1', 'Migrate_2',
       'Migrate_3', 'migration_year', 'migration_interval', 'marriage',
       'employed', 'income_total_m', 'exp_total_m', 'food_exp_m',
       'income_to_home', 'rent_m', 'income_total_m_win', 'exp_total_m_win',
       'food_exp_m_win', 'income_to_home_win', 'rent_m_win',
       'employment_category', 'employment_group', 'workdays_w', 'workhours_d',
       'hours_per_week', 'hours_per_week_filled', 'marriage_year',
       'length_marriage', 'kids_number', 'birth_here', 'Pension_Insurance',
       'Medical_Insurance', 'Work_Insurance', 'Unemploy_Insurance',
       'Maternity_Insurance', 'Housing_Fund', 'Happiness', 'year',
       'lowest_temp(Jan)_b', 'average_temp_b', 'highest_temp(July)_b',
       'precipitation(mm)_b', 'gdp per capita(k)_b', 'unemployment(%)_b',
       'education_budget(10k)_b', 'marriage(10k)_b', 'population(10

In [14]:

# 后续模型保持不变：
# pipe_lr = Pipeline([("prep", preprocess), ("clf", lr)])
# pipe_lgb = Pipeline([("prep", preprocess), ("clf", lgb.LGBMClassifier(...))])

lr = LogisticRegression(
    penalty="l1",
    solver="saga",
    max_iter=200,
    class_weight="balanced",
    C=0.5,
    n_jobs=-1
)

pipe_lr = Pipeline([
    ("prep", preprocess),
    ("clf", lr)
])

pipe_lr.fit(X_tr, y_tr)

def eval_model(model, X, y, name="set"):
    proba = model.predict_proba(X)[:,1]
    roc = roc_auc_score(y, proba)
    ap  = average_precision_score(y, proba)
    prec, rec, thr = precision_recall_curve(y, proba)
    f1s = 2*prec*rec/(prec+rec+1e-12)
    best_idx = np.nanargmax(f1s)
    best_thr = thr[max(best_idx-1, 0)] if len(thr) > 0 else 0.5
    pred = (proba >= best_thr).astype(int)
    f1  = f1_score(y, pred)
    print(f"[{name}] ROC-AUC={roc:.4f}  PR-AUC={ap:.4f}  F1@best={f1:.4f}  thr={best_thr:.3f}")
    print(classification_report(y, pred, digits=3))
    return best_thr, proba

print("=== Validation (LR) ===")
best_thr_lr, _ = eval_model(pipe_lr, X_va, y_va, "valid")
print("=== Test (LR) ===")
_ = eval_model(pipe_lr, X_te, y_te, "test")

=== Validation (LR) ===
[valid] ROC-AUC=0.6181  PR-AUC=0.2320  F1@best=0.3310  thr=0.462
              precision    recall  f1-score   support

           0      0.909     0.457     0.608      6255
           1      0.212     0.761     0.331      1199

    accuracy                          0.506      7454
   macro avg      0.560     0.609     0.469      7454
weighted avg      0.797     0.506     0.563      7454

=== Test (LR) ===
[test] ROC-AUC=0.6172  PR-AUC=0.2438  F1@best=0.3223  thr=0.457
              precision    recall  f1-score   support

           0      0.908     0.409     0.564      6255
           1      0.203     0.785     0.322      1199

    accuracy                          0.469      7454
   macro avg      0.556     0.597     0.443      7454
weighted avg      0.795     0.469     0.525      7454



c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [15]:
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype

RANDOM_STATE = 42

# 0) 构造白名单（与上方“核心特征”一致；按你实际列名调整）
must_keep = [
    "age","male","is_han","rural",
    "high_school","junior_college","bachelor","graduate",
    "employed","employment_group","hours_per_week_filled",
    "log1p_income_total_m_win","log1p_exp_total_m_win","log1p_food_exp_m_win","log1p_income_to_home_win","log1p_rent_m_win",
    "migration_interval",
    "manageable_income_per_capita_b","gdp per capita(k)_b","unemployment(%)_b","population(10k)_b",
    "delta_log_gdp_bj_vs_origin","delta_log_gdp_sh_vs_origin",
    "dist_to_bj","dist_to_sh","dist_bj_minus_sh","dist_min_bj_sh","dist_ratio_bj_over_sh",
    # 如保留 year：添加 "year"
]

# 1) 初步候选集合（去掉标签和键）
label = "y_bjsh"
exclude_cols = {label, "hs_residence", "gdp_bj", "gdp_sh", "log_gdp_before", "log_gdp_bj", "log_gdp_sh"}
cand = [c for c in feature_cols if c not in exclude_cols]

# 2) 规则筛（在全量或抽样上做，建议抽样以提速）
sel_df = df[cand + [label]].sample(n=min(300000, len(df)), random_state=RANDOM_STATE)

# 2.1 删除高缺失
na_rate = sel_df[cand].isna().mean()
drop_high_na = na_rate[na_rate > 0.6].index.tolist()
cand = [c for c in cand if c not in drop_high_na]

# 2.2 稀有类别合并（以 employment_group 为例）
if "employment_group" in cand:
    vc = sel_df["employment_group"].value_counts(dropna=False)
    rare = vc[vc < 500].index  # 频数阈值可调
    sel_df["employment_group"] = sel_df["employment_group"].where(~sel_df["employment_group"].isin(rare), "other")

# 2.3 近零方差（数值列）
num_cols_all = [c for c in cand if is_numeric_dtype(sel_df[c])]
nzv = []
for c in num_cols_all:
    s = sel_df[c].dropna()
    if len(s) == 0: 
        nzv.append(c); 
        continue
    # 若>99% 的值落在极小范围，可视为近零方差
    if s.std(ddof=0) == 0 or (s.quantile(0.99) - s.quantile(0.01) == 0):
        nzv.append(c)
cand = [c for c in cand if c not in nzv]

# 2.4 数值高相关去冗余（Spearman |rho|>0.9）
num_cols = [c for c in cand if is_numeric_dtype(sel_df[c])]
if len(num_cols) > 1:
    corr = sel_df[num_cols].corr(method="spearman").abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_corr = [column for column in upper.columns if any(upper[column] > 0.9)]
    cand = [c for c in cand if c not in drop_corr or c in must_keep]  # must_keep 优先

# 3) 模型驱动筛选：LightGBM + Permutation Importance（在抽样的训练/验证拆分上）
from sklearn.model_selection import train_test_split
X = sel_df[cand].copy()
y = sel_df[label].astype(int).values

# 按 dtype 动态划分数值/类别
obj_cols = X.select_dtypes(include=["object"]).columns.tolist()
# 把“像数值”的字符串转数值（>=90% 可转则转换）
def coerce_numeric_like(dfin, cols, min_numeric_rate=0.9):
    for c in cols:
        s = dfin[c].astype(str).str.replace(",", "").str.strip()
        s_num = pd.to_numeric(s, errors="coerce")
        if s_num.notna().mean() >= min_numeric_rate:
            dfin[c] = s_num
coerce_numeric_like(X, obj_cols)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
# 确保 employment_group 进类别通道
for must_cat in ["employment_group"]:
    if must_cat in X.columns and must_cat not in cat_cols:
        if must_cat in num_cols: num_cols.remove(must_cat)
        cat_cols.append(must_cat)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import lightgbm as lgb
from sklearn.inspection import permutation_importance

# 兼容不同 sklearn 版本的稠密 OHE
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess_sel = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", ohe)]), cat_cols)
    ],
    remainder="drop"
)

X_trs, X_vas, y_trs, y_vas = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

pipe_lgb_sel = Pipeline([
    ("prep", preprocess_sel),
    ("clf", lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.05, num_leaves=64,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        n_jobs=-1, scale_pos_weight=(1 - y_trs.mean())/max(y_trs.mean(),1e-6),
        eval_metric="auc", verbosity=-1
    ))
])

pipe_lgb_sel.fit(X_trs, y_trs)

# 在验证集上做置换重要性（对“原始列”进行打乱，结果可直接按原始列名排序）
perm = permutation_importance(pipe_lgb_sel, X_vas, y_vas, n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1)
imp_df = pd.DataFrame({"feature": X.columns, "importance_mean": perm.importances_mean}).sort_values("importance_mean", ascending=False)

# 选 Top-N 或 importance>0
TOP_N = 40
auto_keep = imp_df.query("importance_mean > 0").head(TOP_N)["feature"].tolist()
selected_features = sorted(set(auto_keep).union(set(must_keep)).intersection(set(feature_cols)))

print(f"[info] 自动保留 {len(auto_keep)} 列；最终（含白名单并集）特征数：{len(selected_features)}")

c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[info] 自动保留 37 列；最终（含白名单并集）特征数：50


In [16]:

# 4) 用筛选后的特征重训最终模型
# 4.1 重建前处理器（用全量训练集），并分别训练 LR 和 LGBM（LGBM 用 callbacks 早停）
X_full = df[selected_features].copy()
y_full = df[label].astype(int).values

# 划分全量数据（随机打乱）
from sklearn.model_selection import train_test_split
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_full, y_full, test_size=0.30, random_state=RANDOM_STATE, stratify=y_full)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_tmp)

# 动态类型划分（全量）
obj_cols = X_tr.select_dtypes(include=["object"]).columns.tolist()
coerce_numeric_like(X_tr, obj_cols); coerce_numeric_like(X_va, obj_cols); coerce_numeric_like(X_te, obj_cols)

num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_tr.columns if c not in num_cols]
for must_cat in ["employment_group"]:
    if must_cat in X_tr.columns and must_cat not in cat_cols:
        if must_cat in num_cols: num_cols.remove(must_cat)
        cat_cols.append(must_cat)

try:
    ohe_final = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe_final = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess_final = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", ohe_final)]), cat_cols)
    ],
    remainder="drop"
)

# 4.2 训练 Logistic（L1）
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve, classification_report

pipe_lr = Pipeline([
    ("prep", preprocess_final),
    ("clf", LogisticRegression(penalty="l1", solver="saga", max_iter=200, class_weight="balanced", C=0.5, n_jobs=-1))
]).fit(X_tr, y_tr)

def eval_model(model, X, y, name):
    proba = model.predict_proba(X)[:,1]
    roc = roc_auc_score(y, proba); ap = average_precision_score(y, proba)
    prec, rec, thr = precision_recall_curve(y, proba)
    f1s = 2*prec*rec/(prec+rec+1e-12); best_idx = np.nanargmax(f1s)
    best_thr = thr[max(best_idx-1, 0)] if len(thr) > 0 else 0.5
    pred = (proba >= best_thr).astype(int); f1 = f1_score(y, pred)
    print(f"[{name}] ROC-AUC={roc:.4f}  PR-AUC={ap:.4f}  F1@best={f1:.4f}  thr={best_thr:.3f}")
    print(classification_report(y, pred, digits=3))
    return best_thr, proba

print("=== Validation (LR) ==="); best_thr_lr, _ = eval_model(pipe_lr, X_va, y_va, "valid")
print("=== Test (LR) ==="); _ = eval_model(pipe_lr, X_te, y_te, "test")

=== Validation (LR) ===
[valid] ROC-AUC=0.6083  PR-AUC=0.2239  F1@best=0.3098  thr=0.516
              precision    recall  f1-score   support

           0      0.899     0.379     0.533      6255
           1      0.193     0.777     0.310      1199

    accuracy                          0.443      7454
   macro avg      0.546     0.578     0.421      7454
weighted avg      0.785     0.443     0.497      7454

=== Test (LR) ===
[test] ROC-AUC=0.6066  PR-AUC=0.2296  F1@best=0.3061  thr=0.513
              precision    recall  f1-score   support

           0      0.910     0.291     0.441      6255
           1      0.187     0.849     0.306      1199

    accuracy                          0.381      7454
   macro avg      0.548     0.570     0.373      7454
weighted avg      0.793     0.381     0.419      7454



c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [17]:
# 10) 构建预处理（按 dtype 动态划分数值/类别）
def coerce_numeric_like(df_in, cols, rate=0.9):
    for c in cols:
        s = df_in[c].astype(str).str.replace(",", "").str.strip()
        s_num = pd.to_numeric(s, errors="coerce")
        if s_num.notna().mean() >= rate:
            df_in[c] = s_num

obj_cols = X_tr.select_dtypes(include=["object"]).columns.tolist()
coerce_numeric_like(X_tr, obj_cols); coerce_numeric_like(X_va, obj_cols); coerce_numeric_like(X_te, obj_cols)

num_cols = X_tr.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_tr.columns if c not in num_cols]
if "employment_group" in X_tr.columns and "employment_group" not in cat_cols:
    if "employment_group" in num_cols: num_cols.remove("employment_group")
    cat_cols.append("employment_group")

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", ohe)]), cat_cols)
    ],
    remainder="drop"
)

# 11) 评估函数
def eval_model(clf, name, Xv, yv):
    proba = clf.predict_proba(Xv)[:,1]
    roc = roc_auc_score(yv, proba)
    ap  = average_precision_score(yv, proba)
    prec, rec, thr = precision_recall_curve(yv, proba)
    f1s = 2*prec*rec/(prec+rec+1e-12)
    best_idx = np.nanargmax(f1s)
    best_thr = thr[max(best_idx-1, 0)] if len(thr) > 0 else 0.5
    pred = (proba >= best_thr).astype(int)
    f1  = f1_score(yv, pred)
    print(f"[{name}] ROC-AUC={roc:.4f}  PR-AUC={ap:.4f}  F1@best={f1:.4f}  thr={best_thr:.3f}")
    print(classification_report(yv, pred, digits=3, target_names=["Shanghai","Beijing"]))
    return proba, best_thr

# 12) Logistic Regression（baseline）
logit = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(
        penalty="l2", solver="lbfgs", max_iter=1000,
        class_weight="balanced", n_jobs=-1
    ))
]).fit(X_tr, y_tr)

print("=== Logistic Regression ===")
_ , _ = eval_model(logit, "valid", X_va, y_va)
_ , _ = eval_model(logit, "test",  X_te, y_te)


=== Logistic Regression ===
[valid] ROC-AUC=0.7177  PR-AUC=0.3036  F1@best=0.4030  thr=0.506
              precision    recall  f1-score   support

    Shanghai      0.914     0.690     0.786      6255
     Beijing      0.290     0.661     0.403      1199

    accuracy                          0.685      7454
   macro avg      0.602     0.675     0.595      7454
weighted avg      0.813     0.685     0.725      7454

[test] ROC-AUC=0.7274  PR-AUC=0.3202  F1@best=0.3963  thr=0.495
              precision    recall  f1-score   support

    Shanghai      0.918     0.653     0.763      6255
     Beijing      0.277     0.694     0.396      1199

    accuracy                          0.660      7454
   macro avg      0.597     0.674     0.580      7454
weighted avg      0.815     0.660     0.704      7454



In [18]:
from sklearn.ensemble import RandomForestClassifier
# 13) Random Forest（长一点时间）
rf = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=1000,            # 可调更大以更久训练
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
]).fit(X_tr, y_tr)

print("=== Random Forest ===")
_ , _ = eval_model(rf, "valid", X_va, y_va)
_ , _ = eval_model(rf, "test",  X_te, y_te)

=== Random Forest ===
[valid] ROC-AUC=0.8343  PR-AUC=0.5373  F1@best=0.5276  thr=0.262
              precision    recall  f1-score   support

    Shanghai      0.918     0.876     0.896      6255
     Beijing      0.477     0.590     0.528      1199

    accuracy                          0.830      7454
   macro avg      0.698     0.733     0.712      7454
weighted avg      0.847     0.830     0.837      7454

[test] ROC-AUC=0.8385  PR-AUC=0.5374  F1@best=0.5311  thr=0.236
              precision    recall  f1-score   support

    Shanghai      0.926     0.848     0.885      6255
     Beijing      0.450     0.648     0.531      1199

    accuracy                          0.816      7454
   macro avg      0.688     0.748     0.708      7454
weighted avg      0.850     0.816     0.828      7454



In [ ]:

# 4.3 训练 LightGBM（用 callbacks 早停，先手动 transform 成纯数值）
from scipy import sparse
def to_float32(mat):
    if sparse.issparse(mat):
        return mat.astype(np.float32).tocsr()
    return np.asarray(mat, dtype=np.float32)

preprocess_final.fit(X_tr)
Xtr = to_float32(preprocess_final.transform(X_tr))
Xva = to_float32(preprocess_final.transform(X_va))
Xte = to_float32(preprocess_final.transform(X_te))

import lightgbm as lgb
lgbm = lgb.LGBMClassifier(
    n_estimators=1200, learning_rate=0.05, num_leaves=64,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    scale_pos_weight=(1 - y_tr.mean())/max(y_tr.mean(),1e-6),
    n_jobs=-1, eval_metric="auc", verbosity=-1
)
try:
    from lightgbm import early_stopping, log_evaluation
    callbacks = [early_stopping(stopping_rounds=100), log_evaluation(period=50)]
except Exception:
    from lightgbm.callback import early_stopping as es, log_evaluation as le
    callbacks = [es(stopping_rounds=100), le(period=50)]

lgbm.fit(Xtr, y_tr, eval_set=[(Xva, y_va)], callbacks=callbacks)

def eval_lgbm(name, Xn, yn):
    proba = lgbm.predict_proba(Xn)[:,1]
    roc = roc_auc_score(yn, proba); ap = average_precision_score(yn, proba)
    prec, rec, thr = precision_recall_curve(yn, proba)
    f1s = 2*prec*rec/(prec+rec+1e-12); best_idx = np.nanargmax(f1s)
    best_thr = thr[max(best_idx-1, 0)] if len(thr) > 0 else 0.5
    pred = (proba >= best_thr).astype(int); f1 = f1_score(yn, pred)
    print(f"[{name}] ROC-AUC={roc:.4f}  PR-AUC={ap:.4f}  F1@best={f1:.4f}  thr={best_thr:.3f}")
    print(classification_report(yn, pred, digits=3))
    return proba, best_thr

print("=== Validation (LGBM) ==="); proba_va, best_thr_lgb = eval_lgbm("valid", Xva, y_va)
print("=== Test (LGBM) ==="); proba_te, _ = eval_lgbm("test", Xte, y_te)

Training until validation scores don't improve for 100 rounds
[50]	valid_0's binary_logloss: 0.476816
[100]	valid_0's binary_logloss: 0.470639
Early stopping, best iteration is:
[7]	valid_0's binary_logloss: 0.414102
=== Validation (LGBM) ===
[valid] ROC-AUC=0.8061  PR-AUC=0.4688  F1@best=0.4827  thr=0.287
              precision    recall  f1-score   support

           0      0.921     0.805     0.859      6255
           1      0.387     0.641     0.483      1199

    accuracy                          0.779      7454
   macro avg      0.654     0.723     0.671      7454
weighted avg      0.835     0.779     0.799      7454

=== Test (LGBM) ===
[test] ROC-AUC=0.8084  PR-AUC=0.4646  F1@best=0.4897  thr=0.304
              precision    recall  f1-score   support

           0      0.910     0.865     0.887      6255
           1      0.439     0.553     0.490      1199

    accuracy                          0.815      7454
   macro avg      0.675     0.709     0.688      7454
weighted 

c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


: 

In [30]:
from catboost import CatBoostClassifier, Pool
import numpy as np
import pandas as pd

# 如果你现在的 X_tr/X_va/X_te 是 DataFrame，就直接用；确保只含 selected_features
X_tr_cb = X_tr[selected_features].copy()
X_va_cb = X_va[selected_features].copy()
X_te_cb = X_te[selected_features].copy()

# 指定类别列（CatBoost按列索引或列名都支持）
cat_cols = []
for c in X_tr_cb.columns:
    if X_tr_cb[c].dtype == "object" or str(X_tr_cb[c].dtype).startswith("category"):
        cat_cols.append(c)
# 确保就业组等确实作为类别
if "employment_group" in X_tr_cb.columns and "employment_group" not in cat_cols:
    cat_cols.append("employment_group")

# CatBoost 允许字符串缺失/数值缺失，直接用即可；也可将类别列统一转为字符串
for c in cat_cols:
    X_tr_cb[c] = X_tr_cb[c].astype("string")
    X_va_cb[c] = X_va_cb[c].astype("string")
    X_te_cb[c] = X_te_cb[c].astype("string")

train_pool = Pool(X_tr_cb, label=y_tr, cat_features=cat_cols)
valid_pool = Pool(X_va_cb, label=y_va, cat_features=cat_cols)
test_pool  = Pool(X_te_cb, label=y_te, cat_features=cat_cols)

# 类不平衡：自动平衡或设置 class_weights（用 auto 更省心）
cb = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",                # 优化AUC；你也可改为 'PRAUC'
    iterations=20000,                 # 大迭代，允许更久训练
    learning_rate=0.02,               # 小学习率
    depth=8,
    l2_leaf_reg=5.0,
    random_seed=42,
    auto_class_weights="Balanced",    # 类不平衡
    od_type="Iter",                   # 早停
    od_wait=800,                      # 等待更久再早停
    task_type="CPU"                   # 有GPU可改为 "GPU"
)
cb.fit(train_pool, eval_set=valid_pool, verbose=200)

# 评估
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve, classification_report
def eval_cb(model, pool, y_true, name):
    proba = model.predict_proba(pool)[:,1]
    roc = roc_auc_score(y_true, proba)
    ap  = average_precision_score(y_true, proba)
    prec, rec, thr = precision_recall_curve(y_true, proba)
    f1s = 2*prec*rec/(prec+rec+1e-12)
    best_idx = np.nanargmax(f1s)
    best_thr = thr[max(best_idx-1, 0)] if len(thr) > 0 else 0.5
    pred = (proba >= best_thr).astype(int)
    f1  = f1_score(y_true, pred)
    print(f"[{name}] ROC-AUC={roc:.4f}  PR-AUC={ap:.4f}  F1@best={f1:.4f}  thr={best_thr:.3f}")
    print(classification_report(y_true, pred, digits=3))
    return proba

print("=== Validation (CatBoost) ===")
proba_va_cb = eval_cb(cb, valid_pool, y_va, "valid")
print("=== Test (CatBoost) ===")
proba_te_cb = eval_cb(cb, test_pool, y_te, "test")

0:	test: 0.7328904	best: 0.7328904 (0)	total: 106ms	remaining: 35m 15s
200:	test: 0.8202153	best: 0.8202153 (200)	total: 22.8s	remaining: 37m 28s
400:	test: 0.8302434	best: 0.8302529 (399)	total: 45.3s	remaining: 36m 53s
600:	test: 0.8377170	best: 0.8377170 (600)	total: 1m 8s	remaining: 36m 36s
800:	test: 0.8427744	best: 0.8427744 (800)	total: 1m 30s	remaining: 36m 16s
1000:	test: 0.8446151	best: 0.8446403 (999)	total: 1m 53s	remaining: 35m 57s
1200:	test: 0.8463480	best: 0.8463480 (1200)	total: 2m 16s	remaining: 35m 38s
1400:	test: 0.8473486	best: 0.8474223 (1367)	total: 2m 39s	remaining: 35m 19s
1600:	test: 0.8481009	best: 0.8482122 (1586)	total: 3m 2s	remaining: 34m 59s
1800:	test: 0.8483708	best: 0.8484018 (1749)	total: 3m 25s	remaining: 34m 37s
2000:	test: 0.8488076	best: 0.8489379 (1957)	total: 3m 48s	remaining: 34m 17s
2200:	test: 0.8492340	best: 0.8492611 (2190)	total: 4m 11s	remaining: 33m 56s
2400:	test: 0.8488915	best: 0.8493331 (2230)	total: 4m 34s	remaining: 33m 34s
2600:	

In [28]:
import xgboost as xgb
import numpy as np
from scipy import sparse

# 用你现成的 preprocess_final（ColumnTransformer）统一处理
preprocess_final.fit(X_tr[selected_features])
Xtr = preprocess_final.transform(X_tr[selected_features])
Xva = preprocess_final.transform(X_va[selected_features])
Xte = preprocess_final.transform(X_te[selected_features])

def to_csr32(mat):
    if sparse.issparse(mat):
        return mat.astype(np.float32).tocsr()
    return sparse.csr_matrix(np.asarray(mat, dtype=np.float32))
Xtr = to_csr32(Xtr); Xva = to_csr32(Xva); Xte = to_csr32(Xte)

dtr = xgb.DMatrix(Xtr, label=y_tr)
dva = xgb.DMatrix(Xva, label=y_va)
dte = xgb.DMatrix(Xte, label=y_te)

# 类不平衡权重
pos_ratio = y_tr.mean()
scale_pos_weight = (1 - pos_ratio) / max(pos_ratio, 1e-6)

params = {
    "objective": "binary:logistic",
    "eval_metric": ["auc","aucpr"],   # 同时看 ROC-AUC 与 PR-AUC
    "eta": 0.03,
    "max_depth": 8,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "gamma": 0.0,
    "tree_method": "hist",            # 有GPU可改 'gpu_hist'
    "scale_pos_weight": scale_pos_weight,
    "nthread": -1,
}

watchlist = [(dtr, "train"), (dva, "valid")]
bst = xgb.train(
    params,
    dtr,
    num_boost_round=20000,            # 大轮数，允许更久训练
    evals=watchlist,
    early_stopping_rounds=800,        # 更耐心的早停
    verbose_eval=200
)

# 评估
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve, classification_report
def eval_xgb(model, dmat, y_true, name):
    proba = model.predict(dmat, iteration_range=(0, model.best_iteration+1))
    roc = roc_auc_score(y_true, proba)
    ap  = average_precision_score(y_true, proba)
    prec, rec, thr = precision_recall_curve(y_true, proba)
    f1s = 2*prec*rec/(prec+rec+1e-12)
    best_idx = np.nanargmax(f1s)
    best_thr = thr[max(best_idx-1, 0)] if len(thr) > 0 else 0.5
    pred = (proba >= best_thr).astype(int)
    f1  = f1_score(y_true, pred)
    print(f"[{name}] ROC-AUC={roc:.4f}  PR-AUC={ap:.4f}  F1@best={f1:.4f}  thr={best_thr:.3f}")
    print(classification_report(y_true, pred, digits=3))
    return proba

print("=== Validation (XGB) ===")
proba_va_xgb = eval_xgb(bst, dva, y_va, "valid")
print("=== Test (XGB) ===")
proba_te_xgb = eval_xgb(bst, dte, y_te, "test")

[0]	train-auc:0.80365	train-aucpr:0.46628	valid-auc:0.75781	valid-aucpr:0.39770
[200]	train-auc:0.93834	train-aucpr:0.76776	valid-auc:0.84286	valid-aucpr:0.54223
[400]	train-auc:0.96995	train-aucpr:0.86515	valid-auc:0.84869	valid-aucpr:0.55778
[600]	train-auc:0.98623	train-aucpr:0.93026	valid-auc:0.85002	valid-aucpr:0.56271
[800]	train-auc:0.99387	train-aucpr:0.96701	valid-auc:0.85071	valid-aucpr:0.56415
[1000]	train-auc:0.99733	train-aucpr:0.98532	valid-auc:0.85108	valid-aucpr:0.56562
[1200]	train-auc:0.99889	train-aucpr:0.99383	valid-auc:0.85065	valid-aucpr:0.56545
[1400]	train-auc:0.99962	train-aucpr:0.99789	valid-auc:0.85005	valid-aucpr:0.56557
[1600]	train-auc:0.99986	train-aucpr:0.99924	valid-auc:0.84966	valid-aucpr:0.56590
[1800]	train-auc:0.99996	train-aucpr:0.99977	valid-auc:0.84978	valid-aucpr:0.56691
[2000]	train-auc:0.99999	train-aucpr:0.99995	valid-auc:0.84955	valid-aucpr:0.56691
[2200]	train-auc:1.00000	train-aucpr:0.99999	valid-auc:0.84878	valid-aucpr:0.56628
[2400]	trai

In [29]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from pandas.api.types import is_numeric_dtype
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve, classification_report

RANDOM_STATE = 42
label = "y_bjsh"

# 1) 划分全局训练/测试（随机、分层）
X_full = df[selected_features].copy()
y_full = df[label].astype(int).values

X_tr_all, X_te_hold, y_tr_all, y_te_hold = train_test_split(
    X_full, y_full, test_size=0.15, random_state=RANDOM_STATE, stratify=y_full
)

# 2) 构建每折内的前处理（数值：median；类别：most_frequent+OHE；按 dtype 动态识别）
def build_preprocess(X):
    # 把“像数值”的object尝试强转
    obj_cols = X.select_dtypes(include=["object"]).columns.tolist()
    for c in obj_cols:
        s = X[c].astype(str).str.replace(",", "").str.strip()
        s_num = pd.to_numeric(s, errors="coerce")
        if s_num.notna().mean() >= 0.9:
            X[c] = s_num
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]
    # 确保 employment_group 走类别通道
    if "employment_group" in X.columns and "employment_group" not in cat_cols:
        if "employment_group" in num_cols:
            num_cols.remove("employment_group")
        cat_cols.append("employment_group")
    # 兼容不同 sklearn 版本的 OHE 稠密输出
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    preprocess = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), num_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", ohe)]), cat_cols)
        ],
        remainder="drop"
    )
    return preprocess

# 3) 三个基学习器的训练接口
from scipy import sparse
def to_float32(mat):
    if sparse.issparse(mat):
        return mat.astype(np.float32).tocsr()
    return np.asarray(mat, dtype=np.float32)

# LightGBM
import lightgbm as lgb
def fit_pred_lgbm(X_tr, y_tr, X_va, y_va, X_te):
    preprocess = build_preprocess(X_tr.copy())
    preprocess.fit(X_tr)
    Xtr = to_float32(preprocess.transform(X_tr))
    Xva = to_float32(preprocess.transform(X_va))
    Xte = to_float32(preprocess.transform(X_te))
    pos_ratio = y_tr.mean()
    scale_pos_weight = (1 - pos_ratio) / max(pos_ratio, 1e-6)
    model = lgb.LGBMClassifier(
        n_estimators=5000,           # 大迭代（更久）
        learning_rate=0.03,         # 小学习率
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
        eval_metric="auc",
        verbosity=-1
    )
    # 兼容 v4 回调
    try:
        from lightgbm import early_stopping, log_evaluation
        callbacks = [early_stopping(stopping_rounds=500), log_evaluation(period=100)]
    except Exception:
        from lightgbm.callback import early_stopping as es, log_evaluation as le
        callbacks = [es(stopping_rounds=500), le(period=100)]
    model.fit(Xtr, y_tr, eval_set=[(Xva, y_va)], callbacks=callbacks)
    p_va = model.predict_proba(Xva)[:,1]
    p_te = model.predict_proba(Xte)[:,1]
    return p_va, p_te

# XGBoost
import xgboost as xgb
def fit_pred_xgb(X_tr, y_tr, X_va, y_va, X_te):
    preprocess = build_preprocess(X_tr.copy())
    preprocess.fit(X_tr)
    Xtr = preprocess.transform(X_tr)
    Xva = preprocess.transform(X_va)
    Xte = preprocess.transform(X_te)
    # 转 CSR32
    if not sparse.issparse(Xtr):
        Xtr = sparse.csr_matrix(np.asarray(Xtr, dtype=np.float32))
        Xva = sparse.csr_matrix(np.asarray(Xva, dtype=np.float32))
        Xte = sparse.csr_matrix(np.asarray(Xte, dtype=np.float32))
    else:
        Xtr = Xtr.astype(np.float32).tocsr()
        Xva = Xva.astype(np.float32).tocsr()
        Xte = Xte.astype(np.float32).tocsr()
    dtr = xgb.DMatrix(Xtr, label=y_tr)
    dva = xgb.DMatrix(Xva, label=y_va)
    dte = xgb.DMatrix(Xte, label=y_te_hold)  # 标签无关，仅占位
    pos_ratio = y_tr.mean()
    scale_pos_weight = (1 - pos_ratio) / max(pos_ratio, 1e-6)
    params = {
        "objective": "binary:logistic",
        "eval_metric": ["auc","aucpr"],
        "eta": 0.03,
        "max_depth": 8,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "gamma": 0.0,
        "tree_method": "hist",   # 有 GPU 可改 'gpu_hist'
        "scale_pos_weight": scale_pos_weight,
        "nthread": -1,
    }
    watchlist = [(dtr,"train"), (dva,"valid")]
    bst = xgb.train(
        params, dtr,
        num_boost_round=20000,           # 大轮数
        evals=watchlist,
        early_stopping_rounds=1000,
        verbose_eval=False
    )
    p_va = bst.predict(dva, iteration_range=(0, bst.best_iteration+1))
    p_te = bst.predict(dte, iteration_range=(0, bst.best_iteration+1))
    return p_va, p_te

# CatBoost
from catboost import CatBoostClassifier, Pool
def fit_pred_cat(X_tr, y_tr, X_va, y_va, X_te):
    # CatBoost 可直接吃 DataFrame；指定类别列
    Xtr = X_tr.copy(); Xva = X_va.copy(); Xte_ = X_te.copy()
    cat_cols = []
    for c in Xtr.columns:
        if Xtr[c].dtype == "object" or str(Xtr[c].dtype).startswith("string") or str(Xtr[c].dtype).startswith("category"):
            cat_cols.append(c)
    if "employment_group" in Xtr.columns and "employment_group" not in cat_cols:
        cat_cols.append("employment_group")
    # 统一类别为字符串，避免稀有/缺失问题
    for c in cat_cols:
        Xtr[c] = Xtr[c].astype("string")
        Xva[c] = Xva[c].astype("string")
        Xte_[c] = Xte_[c].astype("string")
    train_pool = Pool(Xtr, label=y_tr, cat_features=cat_cols)
    valid_pool = Pool(Xva, label=y_va, cat_features=cat_cols)
    test_pool  = Pool(Xte_, cat_features=cat_cols)
    cb = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",             # 用 Logloss 训练；AUC 作附加指标
        custom_metric=["AUC","PRAUC"],
        iterations=30000,                  # 更久
        learning_rate=0.02,
        depth=8,
        l2_leaf_reg=6.0,
        random_seed=RANDOM_STATE,
        auto_class_weights="Balanced",
        od_type="Iter",
        od_wait=1200,
        task_type="CPU"                    # 有GPU可改 "GPU"
    )
    cb.fit(train_pool, eval_set=valid_pool, verbose=False)
    # 训练后按AUC选最优迭代并裁剪（即便GPU也可）
    try:
        auc_curve = cb.eval_metrics(valid_pool, metrics=["AUC"], ntree_start=1, ntree_end=cb.tree_count_, eval_period=1)["AUC"]
        best_iter = int(np.argmax(auc_curve) + 1)
        cb.shrink(ntree_end=best_iter)
    except Exception:
        pass
    p_va = cb.predict_proba(valid_pool)[:,1]
    p_te = cb.predict_proba(test_pool)[:,1]
    return p_va, p_te

# 4) 5折 OOF 生成一级特征
K = 5
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=RANDOM_STATE)

oof_lgb = np.zeros(len(X_tr_all))
oof_xgb = np.zeros(len(X_tr_all))
oof_cat = np.zeros(len(X_tr_all))

test_pred_lgb = np.zeros(len(X_te_hold))
test_pred_xgb = np.zeros(len(X_te_hold))
test_pred_cat = np.zeros(len(X_te_hold))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_all, y_tr_all), 1):
    X_tr, X_va = X_tr_all.iloc[tr_idx], X_tr_all.iloc[va_idx]
    y_tr, y_va = y_tr_all[tr_idx], y_tr_all[va_idx]

    # LGBM
    pva, pte = fit_pred_lgbm(X_tr, y_tr, X_va, y_va, X_te_hold)
    oof_lgb[va_idx] = pva
    test_pred_lgb += pte / K

    # XGB
    pva, pte = fit_pred_xgb(X_tr, y_tr, X_va, y_va, X_te_hold)
    oof_xgb[va_idx] = pva
    test_pred_xgb += pte / K

    # CatBoost
    pva, pte = fit_pred_cat(X_tr, y_tr, X_va, y_va, X_te_hold)
    oof_cat[va_idx] = pva
    test_pred_cat += pte / K

    print(f"fold {fold}/{K} done")

# 5) 二级学习器（元模型）：逻辑回归（也可换 LGBM 小模型）
from sklearn.linear_model import LogisticRegression

X_meta_tr = np.vstack([oof_lgb, oof_xgb, oof_cat]).T
X_meta_te = np.vstack([test_pred_lgb, test_pred_xgb, test_pred_cat]).T

meta = LogisticRegression(
    penalty="l2", solver="lbfgs", max_iter=1000, class_weight="balanced"
).fit(X_meta_tr, y_tr_all)

proba_te = meta.predict_proba(X_meta_te)[:,1]

# 6) 评估（测试集）
def eval_proba(y_true, proba, name):
    roc = roc_auc_score(y_true, proba)
    ap  = average_precision_score(y_true, proba)
    prec, rec, thr = precision_recall_curve(y_true, proba)
    f1s = 2*prec*rec/(prec+rec+1e-12)
    best_idx = np.nanargmax(f1s)
    best_thr = thr[max(best_idx-1, 0)] if len(thr) > 0 else 0.5
    pred = (proba >= best_thr).astype(int)
    f1  = f1_score(y_true, pred)
    print(f"[{name}] ROC-AUC={roc:.4f}  PR-AUC={ap:.4f}  F1@best={f1:.4f} thr={best_thr:.3f}")
    print(classification_report(y_true, pred, digits=3))
    return proba, best_thr

print("=== Test (Stacking) ===")
_ , _ = eval_proba(y_te_hold, proba_te, "stack")

Training until validation scores don't improve for 500 rounds
[100]	valid_0's binary_logloss: 0.47231
[200]	valid_0's binary_logloss: 0.454112
[300]	valid_0's binary_logloss: 0.437538
[400]	valid_0's binary_logloss: 0.424254
[500]	valid_0's binary_logloss: 0.412305
[600]	valid_0's binary_logloss: 0.402809
[700]	valid_0's binary_logloss: 0.394396
[800]	valid_0's binary_logloss: 0.387777
[900]	valid_0's binary_logloss: 0.381717
[1000]	valid_0's binary_logloss: 0.376382
[1100]	valid_0's binary_logloss: 0.371304
[1200]	valid_0's binary_logloss: 0.366713
[1300]	valid_0's binary_logloss: 0.362714
[1400]	valid_0's binary_logloss: 0.359923
[1500]	valid_0's binary_logloss: 0.356943
[1600]	valid_0's binary_logloss: 0.354105
[1700]	valid_0's binary_logloss: 0.351719
[1800]	valid_0's binary_logloss: 0.34938
[1900]	valid_0's binary_logloss: 0.347866
[2000]	valid_0's binary_logloss: 0.346457
[2100]	valid_0's binary_logloss: 0.34561
[2200]	valid_0's binary_logloss: 0.34506
[2300]	valid_0's binary_log

c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


fold 1/5 done
Training until validation scores don't improve for 500 rounds
[100]	valid_0's binary_logloss: 0.480346
[200]	valid_0's binary_logloss: 0.462722
[300]	valid_0's binary_logloss: 0.444863
[400]	valid_0's binary_logloss: 0.43137
[500]	valid_0's binary_logloss: 0.420154
Early stopping, best iteration is:
[11]	valid_0's binary_logloss: 0.415196


c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


fold 2/5 done
Training until validation scores don't improve for 500 rounds
[100]	valid_0's binary_logloss: 0.47047
[200]	valid_0's binary_logloss: 0.453027
[300]	valid_0's binary_logloss: 0.436194
[400]	valid_0's binary_logloss: 0.423428
[500]	valid_0's binary_logloss: 0.412627
[600]	valid_0's binary_logloss: 0.403596
[700]	valid_0's binary_logloss: 0.395453
[800]	valid_0's binary_logloss: 0.388765
[900]	valid_0's binary_logloss: 0.38294
[1000]	valid_0's binary_logloss: 0.377324
[1100]	valid_0's binary_logloss: 0.372395
[1200]	valid_0's binary_logloss: 0.367912
[1300]	valid_0's binary_logloss: 0.364376
[1400]	valid_0's binary_logloss: 0.360877
[1500]	valid_0's binary_logloss: 0.357966
[1600]	valid_0's binary_logloss: 0.35573
[1700]	valid_0's binary_logloss: 0.35345
[1800]	valid_0's binary_logloss: 0.35133
[1900]	valid_0's binary_logloss: 0.350315
[2000]	valid_0's binary_logloss: 0.349272
[2100]	valid_0's binary_logloss: 0.348116
[2200]	valid_0's binary_logloss: 0.346777
[2300]	valid_0

...

fold 3/5 done
Training until validation scores don't improve for 500 rounds
[100]	valid_0's binary_logloss: 0.472994
[200]	valid_0's binary_logloss: 0.457976
[300]	valid_0's binary_logloss: 0.440746
[400]	valid_0's binary_logloss: 0.428143
[500]	valid_0's binary_logloss: 0.417326
Early stopping, best iteration is:
[12]	valid_0's binary_logloss: 0.413439


c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


fold 4/5 done
Training until validation scores don't improve for 500 rounds
[100]	valid_0's binary_logloss: 0.47794
[200]	valid_0's binary_logloss: 0.457959
[300]	valid_0's binary_logloss: 0.439907
[400]	valid_0's binary_logloss: 0.42688
[500]	valid_0's binary_logloss: 0.415809
[600]	valid_0's binary_logloss: 0.406226
[700]	valid_0's binary_logloss: 0.398637
[800]	valid_0's binary_logloss: 0.392075
[900]	valid_0's binary_logloss: 0.386534
[1000]	valid_0's binary_logloss: 0.381159
[1100]	valid_0's binary_logloss: 0.376378
[1200]	valid_0's binary_logloss: 0.372201
[1300]	valid_0's binary_logloss: 0.368514
[1400]	valid_0's binary_logloss: 0.365104
[1500]	valid_0's binary_logloss: 0.362534
[1600]	valid_0's binary_logloss: 0.36048
[1700]	valid_0's binary_logloss: 0.358511
[1800]	valid_0's binary_logloss: 0.356734
[1900]	valid_0's binary_logloss: 0.355374
[2000]	valid_0's binary_logloss: 0.354445
[2100]	valid_0's binary_logloss: 0.353564
[2200]	valid_0's binary_logloss: 0.352898
[2300]	valid

c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


fold 5/5 done
=== Test (Stacking) ===
[stack] ROC-AUC=0.8593  PR-AUC=0.5809  F1@best=0.5483 thr=0.610
              precision    recall  f1-score   support

           0      0.931     0.851     0.889      6255
           1      0.463     0.672     0.548      1199

    accuracy                          0.822      7454
   macro avg      0.697     0.761     0.719      7454
weighted avg      0.856     0.822     0.834      7454



In [14]:
import pandas as pd
import numpy as np
import re

# ========== 1) 读入主表 ==========
df = pd.read_csv("panel_data_1111/after_merge_1111.csv")

# 样本范围：仅跨省流动
df = df[df["Migrate"] == 1].copy()
df = df[df["hs_residence"] != df["pro_code"]].copy()
if "Migrate_1" in df.columns:
    df = df[df["Migrate_1"] == 1].copy()

# 标签：是否迁往北京/上海
df["y_bjsh"] = df["pro_code"].isin([11, 31]).astype(int)

C:\Users\yunzh\AppData\Local\Temp\ipykernel_19716\3750782441.py:6: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("panel_data_1111/after_merge_1111.csv")


In [15]:

# 丢弃泄漏列
drop_cols_leak = [
    "pro_code","pro_name","city_clean","is_beijing","is_shanghai",
    "pro_name_true","English_name","gdp_after_move","migration_distance_km"
]
drop_cols_leak += [c for c in df.columns if c.endswith("_a")]  # 所有目的地 after 列
df = df.drop(columns=[c for c in drop_cols_leak if c in df.columns])

# ========== 2) 基础个人特征 ==========
keep_personal = [
    "male","age","is_han","rural",
    "high_school","junior_college","bachelor","graduate",
    "marriage","length_marriage","kids_number","birth_here",
    "Happiness","employed","employment_group","hours_per_week_filled",
    "income_total_m_win","exp_total_m_win","food_exp_m_win","income_to_home_win","rent_m_win",
    "year","migration_year","migration_interval","hs_residence"
]
keep_personal = [c for c in keep_personal if c in df.columns]

# 原籍宏观（全部 *_b）
macro_b_cols = [c for c in df.columns if c.endswith("_b")]

df = df[keep_personal + macro_b_cols + ["y_bjsh"]].copy()

# 金额类做 log1p
for c in ["income_total_m_win","exp_total_m_win","food_exp_m_win","income_to_home_win","rent_m_win"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
        df[f"log1p_{c}"] = np.log1p(df[c])

c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [16]:
df

,male,age,is_han,rural,high_school,junior_college,bachelor,graduate,marriage,length_marriage,...,population(10k)_b,Medical technicians per 10k_b,road_length_per_10K (km)_b,manageable_income_per_capita_b,y_bjsh,log1p_income_total_m_win,log1p_exp_total_m_win,log1p_food_exp_m_win,log1p_income_to_home_win,log1p_rent_m_win
0,1,35,1,1,1,0,0,0,0,0,...,325.0,41.0,55.086354,10730.0,1,7.601402,6.685861,6.216606,9.210440,0.000000
1,1,32,1,1,1,0,0,0,0,0,...,9645.0,52.0,3.921530,15695.0,1,7.244942,6.803505,6.216606,0.000000,0.000000
2,0,38,1,1,1,0,0,0,1,18,...,2449.0,62.0,16.694917,20559.0,1,8.071219,6.685861,5.303305,8.294300,0.000000
3,0,24,1,1,0,1,0,0,0,0,...,2449.0,62.0,16.694917,20559.0,1,7.313887,6.685861,6.216606,6.908755,0.000000
4,0,21,0,1,1,0,0,0,0,0,...,4653.0,44.0,5.698520,13772.0,1,7.438972,5.707110,4.615121,9.105091,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
892282,0,25,1,0,1,0,0,0,0,2013,...,4653.0,44.0,5.698520,13772.0,0,8.294300,7.824446,0.000000,0.000000,6.216606
892284,1,21,0,0,1,0,0,0,0,2013,...,2325.0,67.0,29.946334,15097.0,0,7.313887,6.685861,0.000000,0.000000,0.000000
892285,1,20,1,0,1,0,0,0,0,2013,...,2325.0,67.0,29.946334,15097.0,0,7.783641,6.685861,0.000000,0.000000,0.000000
892286,1,45,1,0,1,0,0,0,0,2014,...,2467.0,68.0,15.087278,45966.0,0,8.006701,6.552508,0.000000,0.000000,0.000000


In [17]:

# ========== 3) GDP 年度“拉力”特征（只用 before） ==========
# 读取省-年 GDP 面板，取北京/上海本年的 GDP
# 你可用 china_correct_panel.xlsx（含 pro_code/year/real_GDP），或你自己的 GDP 面板
gdp_panel = pd.read_excel("china_correct_panel.xlsx")  # 包含 pro_code, year, real_GDP（或人均）
# 统一列名
gdp_panel = gdp_panel.rename(columns={"year":"migration_year"})
bj = gdp_panel[gdp_panel["pro_code"] == 11][["migration_year","real_GDP"]].rename(columns={"real_GDP":"gdp_bj"})
sh = gdp_panel[gdp_panel["pro_code"] == 31][["migration_year","real_GDP"]].rename(columns={"real_GDP":"gdp_sh"})

df = df.merge(bj, on="migration_year", how="left")
df = df.merge(sh, on="migration_year", how="left")

# gdp_before_move 如果是“人均 GDP”更好；若是总量，也可先做 log 差
if "gdp_before_move" in df.columns:
    df["log_gdp_before"] = np.log(df["gdp_before_move"].replace({0: np.nan}))
else:
    # 若没有该列，可用 external_data2 的人均收入/可支配收入等替代；此处简化为 NaN
    df["log_gdp_before"] = np.nan

# 构造 log 差
for tgt in ["gdp_bj","gdp_sh"]:
    df[f"log_{tgt}"] = np.log(df[tgt].replace({0: np.nan}))
    df[f"delta_log_{tgt}_vs_origin"] = df[f"log_{tgt}"] - df["log_gdp_before"]

In [19]:
import pandas as pd
import numpy as np

# 0) 读取 distance.xlsx，并做类型统一
dist = pd.read_excel("distance.xlsx")
# 确保代码为整数、距离为浮点
for c in ["hs_residence", "pro_code"]:
    dist[c] = pd.to_numeric(dist[c], errors="coerce").astype("Int64")
dist["migration_distance_km"] = pd.to_numeric(dist["migration_distance_km"], errors="coerce")

# 1) 做“对称闭包”：A→B 距离若缺，尝试用 B→A 补
dist_rev = dist.rename(columns={"hs_residence":"pro_code", "pro_code":"hs_residence"})
dist_symm = pd.concat([dist, dist_rev], ignore_index=True)

# 去重：若同一对 (hs_residence, pro_code) 有多条，取均值或最小值（一般不会）
dist_symm = (dist_symm
             .dropna(subset=["hs_residence","pro_code","migration_distance_km"])
             .groupby(["hs_residence","pro_code"], as_index=False)["migration_distance_km"].mean())

# 2) 构造“原籍→北京/上海”的映射表（不依赖样本真实去向）
bj_code, sh_code = 11, 31

map_bj = (dist_symm[dist_symm["pro_code"] == bj_code]
          .loc[:, ["hs_residence", "migration_distance_km"]]
          .rename(columns={"migration_distance_km":"dist_to_bj"}))
map_sh = (dist_symm[dist_symm["pro_code"] == sh_code]
          .loc[:, ["hs_residence", "migration_distance_km"]]
          .rename(columns={"migration_distance_km":"dist_to_sh"}))

# 给北京、上海自身补 0 距离（若表中没有）
for code, col in [(bj_code, "dist_to_bj"), (sh_code, "dist_to_sh")]:
    if code not in map_bj["hs_residence"].values and col == "dist_to_bj":
        map_bj = pd.concat([map_bj, pd.DataFrame({"hs_residence":[bj_code], "dist_to_bj":[0.0]})], ignore_index=True)
    if code not in map_sh["hs_residence"].values and col == "dist_to_sh":
        map_sh = pd.concat([map_sh, pd.DataFrame({"hs_residence":[sh_code], "dist_to_sh":[0.0]})], ignore_index=True)

# 3) 合并成一个“原籍→京沪距离”字典，供训练时按 hs_residence 合并
dist_bj_sh = map_bj.merge(map_sh, on="hs_residence", how="outer")

# 覆盖检查：31 个省份是否基本齐全
print("原籍到北京距离覆盖省份数：", dist_bj_sh["hs_residence"].nunique())
print(dist_bj_sh.head())

# 4) 保存映射，后续建模随取随用
dist_bj_sh.to_csv("dist_origin_to_bj_sh.csv", index=False, encoding="utf-8-sig")

# 5) 将距离并回你的主表（仅用 hs_residence 合并，不要用 pro_code）
df = pd.read_csv("panel_data_1111/after_merge_1111.csv")
# 确保类型一致
df["hs_residence"] = pd.to_numeric(df["hs_residence"], errors="coerce").astype("Int64")

df = df.merge(dist_bj_sh, on="hs_residence", how="left")

# 6) 可选：派生距离特征（不泄漏，纯原籍决定）
# - 与谁更近（负值表示更近北京，正值表示更近上海）
df["dist_bj_minus_sh"] = df["dist_to_bj"] - df["dist_to_sh"]
# - 最近的那一方的距离
df["dist_min_bj_sh"] = df[["dist_to_bj","dist_to_sh"]].min(axis=1)
# - 相对远近差（比例，避免量纲影响；加1防除零）
df["dist_ratio_bj_over_sh"] = (df["dist_to_bj"] + 1) / (df["dist_to_sh"] + 1)

# 后续在特征清单中加入：
# ["dist_to_bj","dist_to_sh","dist_bj_minus_sh","dist_min_bj_sh","dist_ratio_bj_over_sh"]
# 注意：不要把 migration_distance_km（真实目的地距离）放入模型，否则会泄漏

原籍到北京距离覆盖省份数： 34
   hs_residence  dist_to_bj   dist_to_sh
0            11    0.000000  1092.419688
1            12  126.701976   969.162770
2            13   74.616543  1040.720966
3            14  460.537010  1097.004901
4            15  479.937493  1572.210031
